In [1]:
# ============================================================
# CELL 1 — IMPORTS, CONFIGURATION & BACKUP
# ============================================================
# ALGORITHM NOTE
# ────────────────────────────────────────────────────────────
# sklearn < 1.2 → default = SAMME.R (real-valued probabilities,
# faster convergence, usually better accuracy)
# sklearn ≥ 1.4 → SAMME.R removed; only SAMME is available.
# Since you get an error on algorithm='SAMME.R', you are on
# sklearn ≥ 1.4 — both notebooks effectively run SAMME.
# ────────────────────────────────────────────────────────────

import os
import shutil
import datetime
import numpy as np
import joblib

from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.ensemble import AdaBoostClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.utils.validation import check_is_fitted
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold

# ── DATA PATHS ───────────────────────────────────────────────
X_PATH = r"/home/intern/Downloads/train_X.npy"
Y_PATH = r"/home/intern/Downloads/train_y.npy"
GROUPS_PATH = r"/home/intern/Downloads/train_groups.npy"

# ── MODEL PATHS ──────────────────────────────────────────────
# Create the benchmark-required checkpoints directory
CHECKPOINT_DIR = "checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

# Updated for v2.2
MODEL_SAVE_PATH = os.path.join(CHECKPOINT_DIR, "best_model_v2_2.pt")
MODEL_BACKUP_DIR = "model_backups"

# ── GLOBAL SETTINGS / HYPERPARAMETERS ────────────────────────
RANDOM_STATE = 42

os.makedirs(MODEL_BACKUP_DIR, exist_ok=True)

print("=== ADABOOST + MLP WEAK-LEARNER CONFIGURATION v2.2 ===")
print(f"Feature file : {X_PATH}")
print(f"Label file   : {Y_PATH}")
print(f"Model output : {MODEL_SAVE_PATH}")

if os.path.exists(MODEL_SAVE_PATH):
    timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
    backup_path = os.path.join(
        MODEL_BACKUP_DIR,
        f"best_model_v2_2_{timestamp}.pt"
    )
    shutil.copy2(MODEL_SAVE_PATH, backup_path)
    print(f"Existing model backed up → {backup_path}")
else:
    print("No existing model found — fresh training run.")

=== ADABOOST + MLP WEAK-LEARNER CONFIGURATION v2.2 ===
Feature file : /home/intern/Downloads/train_X.npy
Label file   : /home/intern/Downloads/train_y.npy
Model output : checkpoints/best_model_v2_2.pt
No existing model found — fresh training run.


In [ ]:
# ============================================================
# CELL 2 — LOAD & SPLIT (PATIENT-WISE, NO SMOTE)
# ============================================================
from sklearn.model_selection import GroupShuffleSplit
from collections import Counter
import numpy as np

print("\nLoading feature matrix, labels, and groups...")
X = np.load(X_PATH)
y = np.load(Y_PATH)
# Load the groups file you created with label_assignment.py
groups = np.load(GROUPS_PATH)

print(f"Loaded → X={X.shape}, y={y.shape}, groups={groups.shape}")

unique, counts = np.unique(y, return_counts=True)
print(f"Full dataset class distribution: {dict(zip(unique, counts))}")

# ── SPLIT HYPERPARAMETERS ────────────────────────────────────
TEST_SIZE = 0.10
VAL_SIZE_OF_REMAINING = 0.1111111111  # ~= 10% of full data after 90/10 test split
RANDOM_STATE = 42

# Step 1: Hold out untouched Test Set based on PATIENTS
gss_test = GroupShuffleSplit(n_splits=1, test_size=TEST_SIZE, random_state=RANDOM_STATE)
train_val_idx, test_idx = next(gss_test.split(X, y, groups=groups))

X_temp, y_temp, groups_temp = X[train_val_idx], y[train_val_idx], groups[train_val_idx]
X_test, y_test = X[test_idx], y[test_idx]

# Step 2: Split remaining data into Train + Validation based on PATIENTS
gss_val = GroupShuffleSplit(n_splits=1, test_size=VAL_SIZE_OF_REMAINING, random_state=RANDOM_STATE)
train_idx, val_idx = next(gss_val.split(X_temp, y_temp, groups=groups_temp))

X_train, y_train = X_temp[train_idx], y_temp[train_idx]
X_val, y_val = X_temp[val_idx], y_temp[val_idx]


print(f"\nTraining distribution   : {Counter(y_train)}")
print(f"Validation distribution : {Counter(y_val)}")
print(f"Test distribution       : {Counter(y_test)}")

# Summary counts
print(f"\n{'='*60}")
print("DATA SUMMARY (NO SMOTE)")
print(f"{'='*60}")
print(f"Total training images : {X_train.shape[0]}")
print(f"Validation images     : {X_val.shape[0]}")
print(f"Test images           : {X_test.shape[0]}")
print(f"Features per image    : {X_train.shape[1]}")
print(f"{'='*60}")

In [3]:
# ============================================================
# CELL 3 — MLP WEAK-LEARNER WRAPPER (v2)
# ============================================================
# WHY A WRAPPER?
# ─────────────────────────────────────────────────────────────
# AdaBoostClassifier internally calls estimator.fit(X, y,
# sample_weight=w) to re-focus each round on hard samples.
# sklearn's MLPClassifier.fit() does NOT accept sample_weight,
# so a direct plug-in raises a TypeError. This wrapper converts
# the weight vector into a resampling distribution instead.
# ─────────────────────────────────────────────────────────────

class WeakMLP(BaseEstimator, ClassifierMixin):
    """
    Tiny MLP wrapper for AdaBoostClassifier.
    Handles sample_weight via weighted resampling.
    Adds L2 regularisation, early stopping, and explicit batch size.
    """

    def __init__(
        self,
        hidden_layer_sizes=(32,),
        max_iter=15,
        activation='relu',
        alpha=1e-3,
        batch_size=64,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=5,
        use_class_balance=False,
        random_state=None
    ):
        self.hidden_layer_sizes = hidden_layer_sizes
        self.max_iter = max_iter
        self.activation = activation
        self.alpha = alpha
        self.batch_size = batch_size
        self.early_stopping = early_stopping
        self.validation_fraction = validation_fraction
        self.n_iter_no_change = n_iter_no_change
        self.use_class_balance = use_class_balance
        self.random_state = random_state

    def fit(self, X, y, sample_weight=None):
        # ── RESAMPLING BIAS NOTE ────────────────────────────────────
        # MLPClassifier historically did not support sample_weight.
        # This workaround approximates it by resampling the dataset.
        # Trade-off: It can duplicate high-weight samples and omit low-weight ones,
        # shrinking the effective dataset size, especially in later rounds.
        # ────────────────────────────────────────────────────────────
        rng = np.random.RandomState(self.random_state)

        final_weights = np.ones(len(y), dtype=float)

        # 1) AdaBoost hard-sample emphasis
        if sample_weight is not None:
            boost_w = np.asarray(sample_weight, dtype=float)
            boost_w = boost_w / boost_w.sum()
            final_weights *= boost_w

        # 2) Optional inverse class-frequency emphasis
        if self.use_class_balance:
            class_counts = Counter(y)
            class_weight_map = {
                cls: len(y) / (len(class_counts) * count)
                for cls, count in class_counts.items()
            }
            class_w = np.array([class_weight_map[label] for label in y], dtype=float)
            final_weights *= class_w

        final_weights = final_weights / final_weights.sum()

        idx = rng.choice(len(X), size=len(X), replace=True, p=final_weights)
        X_r, y_r = X[idx], y[idx]

        self.mlp_ = MLPClassifier(
            hidden_layer_sizes=self.hidden_layer_sizes,
            max_iter=self.max_iter,
            activation=self.activation,
            solver='adam',
            alpha=self.alpha,
            batch_size=self.batch_size,
            learning_rate_init=1e-3,
            early_stopping=self.early_stopping,
            validation_fraction=self.validation_fraction,
            n_iter_no_change=self.n_iter_no_change,
            warm_start=False,
            random_state=self.random_state
        )

        self.mlp_.fit(X_r, y_r)
        self.classes_ = self.mlp_.classes_
        # Print final loss to verify convergence
        print(f"  [WeakMLP] trained in {self.mlp_.n_iter_} iters, final loss: {self.mlp_.loss_:.4f}")
        return self

    def predict(self, X):
        check_is_fitted(self, 'mlp_')
        return self.mlp_.predict(X)

    def predict_proba(self, X):
        check_is_fitted(self, 'mlp_')
        return self.mlp_.predict_proba(X)

print("WeakMLP v2 defined — L2 alpha, early stopping, batch_size=64")
print("Architecture options:")
print(" A: (16,) ultra-weak")
print(" B: (32,) weak ← DEFAULT / best so far")
print(" C: (32, 16) mild")

WeakMLP v2 defined — L2 alpha, early stopping, batch_size=64
Architecture options:
 A: (16,) ultra-weak
 B: (32,) weak ← DEFAULT / best so far
 C: (32, 16) mild


In [4]:
# ============================================================
# CELL 4 — FRACTIONAL SEARCH (ARCH B ONLY) + PICK BEST MODEL
# ============================================================
from joblib import Parallel, delayed
from sklearn.metrics import f1_score
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GroupKFold
import numpy as np

# ── FIXED ARCHITECTURE ──────────────────────────────────────
ARCH = 'B'
ARCH_LABEL = "Weak (32,) ← fixed for this search"
HIDDEN_LAYER_SIZES = (32,)

# ── FIXED MLP SETTINGS ──────────────────────────────────────
MLP_ALPHA = 1e-3
MLP_BATCH_SIZE = 64
MLP_ACTIVATION = 'relu'
MLP_EARLY_STOPPING = True
MLP_VALIDATION_FRACTION = 0.1
MLP_N_ITER_NO_CHANGE = 5
USE_CLASS_BALANCE = False

# ── MULTIPLE SEEDS FOR EVALUATION ───────────────────────────
SEEDS = [42]

# ── 9 SEARCH COMBINATIONS ───────────────────────────────────
# 9-combo fractional / Latin-square-style search
SEARCH_COMBINATIONS = [
    {"MLP_MAX_ITER": 50, "N_ESTIMATORS": 100, "LEARNING_RATE": 0.3},
    {"MLP_MAX_ITER": 50, "N_ESTIMATORS": 150, "LEARNING_RATE": 0.5},
    {"MLP_MAX_ITER": 50, "N_ESTIMATORS": 250, "LEARNING_RATE": 0.8},
    {"MLP_MAX_ITER": 50, "N_ESTIMATORS": 100, "LEARNING_RATE": 0.5},
    {"MLP_MAX_ITER": 50, "N_ESTIMATORS": 150, "LEARNING_RATE": 0.8},
    {"MLP_MAX_ITER": 50, "N_ESTIMATORS": 250, "LEARNING_RATE": 0.3},
    {"MLP_MAX_ITER": 50, "N_ESTIMATORS": 100, "LEARNING_RATE": 0.8},
    {"MLP_MAX_ITER": 50, "N_ESTIMATORS": 150, "LEARNING_RATE": 0.3},
    {"MLP_MAX_ITER": 50, "N_ESTIMATORS": 250, "LEARNING_RATE": 0.5},
]

print(f"Selected architecture : {ARCH_LABEL}")
print(f"\n{'='*70}")
print("FRACTIONAL SEARCH CONFIG (PARALLEL CV)")
print(f"{'='*70}")
print(f"total combinations : {len(SEARCH_COMBINATIONS)}")
print(f"eval seeds : {SEEDS}")
print(f"selection metric : Macro F1 Score (GroupKFold CV)")
print(f"{'='*70}")

def evaluate_combo(i, combo, X_cv, y_cv, groups_cv):
    MLP_MAX_ITER = combo["MLP_MAX_ITER"]
    N_ESTIMATORS = combo["N_ESTIMATORS"]
    LEARNING_RATE = combo["LEARNING_RATE"]
    
    gkf = GroupKFold(n_splits=3)
    val_f1_scores = []
    
    # Iterate over CV folds
    for train_idx, val_idx in gkf.split(X_cv, y_cv, groups=groups_cv):
        X_fold_train, X_fold_val = X_cv[train_idx], X_cv[val_idx]
        y_fold_train, y_fold_val = y_cv[train_idx], y_cv[val_idx]
        
        # Scale inside the fold to prevent leakage
        fold_scaler = StandardScaler()
        X_fold_train = fold_scaler.fit_transform(X_fold_train)
        X_fold_val = fold_scaler.transform(X_fold_val)
        
        # Evaluate across seeds
        fold_seed_f1s = []
        for seed in SEEDS:
            base_estimator = WeakMLP(
                hidden_layer_sizes=HIDDEN_LAYER_SIZES,
                max_iter=MLP_MAX_ITER,
                activation=MLP_ACTIVATION,
                alpha=MLP_ALPHA,
                batch_size=MLP_BATCH_SIZE,
                early_stopping=MLP_EARLY_STOPPING,
                validation_fraction=MLP_VALIDATION_FRACTION,
                n_iter_no_change=MLP_N_ITER_NO_CHANGE,
                use_class_balance=USE_CLASS_BALANCE,
                random_state=seed
            )
            
            candidate_model = AdaBoostClassifier(
                estimator=base_estimator,
                n_estimators=N_ESTIMATORS,
                learning_rate=LEARNING_RATE,
                random_state=seed
            )
            
            candidate_model.fit(X_fold_train, y_fold_train)
            y_val_pred = candidate_model.predict(X_fold_val)
            val_f1 = f1_score(y_fold_val, y_val_pred, average='macro')
            fold_seed_f1s.append(val_f1)
            
        val_f1_scores.append(np.mean(fold_seed_f1s))
        
    avg_val_f1 = np.mean(val_f1_scores)
    
    return {
        "combo_no": i,
        "combo": combo,
        "avg_val_f1": avg_val_f1,
        "f1_scores": val_f1_scores
    }

# Run in parallel using joblib
print("Starting parallel CV fractional search...")
results = Parallel(n_jobs=4, verbose=10)(
    delayed(evaluate_combo)(i, combo, X_temp, y_temp, groups_temp) 
    for i, combo in enumerate(SEARCH_COMBINATIONS, start=1)
)

best_result = max(results, key=lambda x: x["avg_val_f1"])
best_combo = best_result["combo"]
MLP_MAX_ITER = best_combo["MLP_MAX_ITER"]
N_ESTIMATORS = best_combo["N_ESTIMATORS"]
LEARNING_RATE = best_combo["LEARNING_RATE"]
chosen = {"label": ARCH_LABEL, "hidden_layer_sizes": HIDDEN_LAYER_SIZES}

print(f"\n{'='*70}")
print("RANKED COMBINATIONS (Macro F1 over CV Folds & Seeds)")
print(f"{'='*70}")
results_sorted = sorted(results, key=lambda x: x["avg_val_f1"], reverse=True)
for rank, r in enumerate(results_sorted, start=1):
    print(
        f"Rank {rank:>2} | "
        f"MAX_ITER={r['combo']['MLP_MAX_ITER']:<2} | "
        f"EST={r['combo']['N_ESTIMATORS']:<3} | "
        f"LR={r['combo']['LEARNING_RATE']:<3} | "
        f"avg_f1={r['avg_val_f1']:.4f} "
        f"(folds: {[round(x,4) for x in r['f1_scores']]})"
    )

print(f"\n{'='*70}")
print("BEST COMBINATION SELECTED")
print(f"{'='*70}")
print(f"MLP_MAX_ITER : {MLP_MAX_ITER}")
print(f"N_ESTIMATORS : {N_ESTIMATORS}")
print(f"LEARNING_RATE : {LEARNING_RATE}")
print(f"Best Avg Validation Macro F1 : {best_result['avg_val_f1']:.4f}")
print(f"{'='*70}")

# Retrain the best combination on the full X_temp (Train + Val)...
print("\nRetraining best combination on the full X_temp (Train + Val)...")
base_estimator = WeakMLP(
    hidden_layer_sizes=HIDDEN_LAYER_SIZES,
    max_iter=MLP_MAX_ITER,
    activation=MLP_ACTIVATION,
    alpha=MLP_ALPHA,
    batch_size=MLP_BATCH_SIZE,
    early_stopping=MLP_EARLY_STOPPING,
    validation_fraction=MLP_VALIDATION_FRACTION,
    n_iter_no_change=MLP_N_ITER_NO_CHANGE,
    use_class_balance=USE_CLASS_BALANCE,
    random_state=42
)
best_model = AdaBoostClassifier(
    estimator=base_estimator,
    n_estimators=N_ESTIMATORS,
    learning_rate=LEARNING_RATE,
    random_state=42
)

from sklearn.pipeline import Pipeline
final_pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("model", best_model)
])
final_pipeline.fit(X_temp, y_temp)

Selected architecture : Weak (32,) ← fixed for this search

FRACTIONAL SEARCH CONFIG (PARALLEL CV)
total combinations : 9
eval seeds : [42]
selection metric : Macro F1 Score (GroupKFold CV)
Starting parallel CV fractional search...


[Parallel(n_jobs=4)]: Using backend LokyBackend with 4 concurrent workers.


  [WeakMLP] trained in 19 iters, final loss: 0.1482
  [WeakMLP] trained in 19 iters, final loss: 0.1482
  [WeakMLP] trained in 19 iters, final loss: 0.1482
  [WeakMLP] trained in 19 iters, final loss: 0.1482
  [WeakMLP] trained in 38 iters, final loss: 0.0248
  [WeakMLP] trained in 36 iters, final loss: 0.0655
  [WeakMLP] trained in 37 iters, final loss: 0.0760
  [WeakMLP] trained in 37 iters, final loss: 0.0760
  [WeakMLP] trained in 40 iters, final loss: 0.0448
  [WeakMLP] trained in 39 iters, final loss: 0.0627
  [WeakMLP] trained in 42 iters, final loss: 0.1030
  [WeakMLP] trained in 42 iters, final loss: 0.0243
  [WeakMLP] trained in 25 iters, final loss: 0.1266
  [WeakMLP] trained in 42 iters, final loss: 0.1030
  [WeakMLP] trained in 40 iters, final loss: 0.0108
  [WeakMLP] trained in 48 iters, final loss: 0.0423
  [WeakMLP] trained in 41 iters, final loss: 0.0457
  [WeakMLP] trained in 38 iters, final loss: 0.0475
  [WeakMLP] trained in 48 iters, final loss: 0.0423
  [WeakMLP] 

/home/intern/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


  [WeakMLP] trained in 44 iters, final loss: 0.0339


...

/home/intern/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


  [WeakMLP] trained in 50 iters, final loss: 0.0486
  [WeakMLP] trained in 46 iters, final loss: 0.0562
  [WeakMLP] trained in 41 iters, final loss: 0.0901
  [WeakMLP] trained in 48 iters, final loss: 0.0614
  [WeakMLP] trained in 38 iters, final loss: 0.0839
  [WeakMLP] trained in 43 iters, final loss: 0.0719


/home/intern/.local/lib/python3.10/site-packages/sklearn/neural_network/_multilayer_perceptron.py:781: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (50) reached and the optimization hasn't converged yet.
  warnings.warn(


  [WeakMLP] trained in 50 iters, final loss: 0.0784
  [WeakMLP] trained in 43 iters, final loss: 0.0714
  [WeakMLP] trained in 44 iters, final loss: 0.0941


,steps,"[('scaler', ...), ('model', ...)]"
,transform_input,None
,memory,None
,verbose,False
,copy,True
,with_mean,True
,with_std,True
,estimator,WeakMLP(max_i...ndom_state=42)
,n_estimators,250
,learning_rate,0.5
,algorithm,'deprecated'


In [5]:
# ============================================================
# CELL 5 — EVALUATE BEST MODEL ON VALIDATION + TEST
# ============================================================

# ── EVALUATION / REPORTING HYPERPARAMETERS ───────────────────
SHOW_FIRST_N_PROB_ROWS = 5
RARE_CLASS_NAMES = [
    'Tissue Density Category 1',
    'Tissue Density Category 4'
]
TARGET_NAMES = [
    'Tissue Density Category 1',
    'Tissue Density Category 2',
    'Tissue Density Category 3',
    'Tissue Density Category 4'
]

# ── PREDICT ON VALIDATION AND TEST ───────────────────────────
print("\nEvaluating BEST selected model on validation set...")
y_val_pred = final_pipeline.predict(X_val)
y_val_proba = final_pipeline.predict_proba(X_val)

print("Evaluating BEST selected model on held-out test set...")
y_test_pred = final_pipeline.predict(X_test)
y_test_proba = final_pipeline.predict_proba(X_test)

# ── METRICS ──────────────────────────────────────────────────
from sklearn.metrics import f1_score
val_acc = accuracy_score(y_val, y_val_pred)
test_acc = accuracy_score(y_test, y_test_pred)
val_macro_f1 = f1_score(y_val, y_val_pred, average="macro")
test_macro_f1 = f1_score(y_test, y_test_pred, average="macro")

val_cm = confusion_matrix(y_val, y_val_pred)
test_cm = confusion_matrix(y_test, y_test_pred)

val_report_dict = classification_report(
    y_val,
    y_val_pred,
    target_names=TARGET_NAMES,
    zero_division=0,
    output_dict=True
)

test_report_dict = classification_report(
    y_test,
    y_test_pred,
    target_names=TARGET_NAMES,
    zero_division=0,
    output_dict=True
)

# ── SUMMARY REPORT ───────────────────────────────────────────
print(f"\n{'='*60}")
print("PERFORMANCE SUMMARY")
print(f"{'='*60}")
print(f"Architecture : {chosen['label']}")
print(f"Best combo selected from validation search:")
print(f" MLP_MAX_ITER : {MLP_MAX_ITER}")
print(f" N_ESTIMATORS : {N_ESTIMATORS}")
print(f" LEARNING_RATE : {LEARNING_RATE}")
print(f" MLP alpha (L2) : {MLP_ALPHA}")
print(f" MLP batch_size : {MLP_BATCH_SIZE}")
print(f"Training images : {len(X_train)}")     # <--- UPDATED
print(f"Validation images : {len(X_val)}")   # <--- UPDATED
print(f"Test images : {len(X_test)}")        # <--- UPDATED
print(f"Validation accuracy : {val_acc:.4f}")
print(f"Test accuracy : {test_acc:.4f}")
print(f"Validation Macro F1 : {val_macro_f1:.4f}")
print(f"Test Macro F1 : {test_macro_f1:.4f}")
print(f"{'='*60}")

# ── VALIDATION REPORT ────────────────────────────────────────
print(f"\n{'='*60}")
print("VALIDATION REPORT")
print(f"{'='*60}")
print(classification_report(
    y_val,
    y_val_pred,
    target_names=TARGET_NAMES,
    zero_division=0
))

print("Validation Confusion Matrix:")
header = " " + " ".join(f"Pred {i+1:>4}" for i in range(4))
print(header)
for i, row in enumerate(val_cm):
    print(f"Actual {i+1} " + " ".join(f"{v:>8d}" for v in row))

print(f"\nFirst {SHOW_FIRST_N_PROB_ROWS} validation class-probability rows:")
print(y_val_proba[:SHOW_FIRST_N_PROB_ROWS])

print(f"\n{'='*60}")
print("VALIDATION RARE-CLASS FOCUS")
print(f"{'='*60}")
for name in RARE_CLASS_NAMES:
    r = val_report_dict[name]
    print(f"{name}")
    print(f" recall : {r['recall']:.4f}")
    print(f" f1-score : {r['f1-score']:.4f}")
    print(f" precision : {r['precision']:.4f}")
print(f"{'='*60}")

# ── TEST REPORT ──────────────────────────────────────────────
print(f"\n{'='*60}")
print("TEST REPORT")
print(f"{'='*60}")
print(classification_report(
    y_test,
    y_test_pred,
    target_names=TARGET_NAMES,
    zero_division=0
))

print("Test Confusion Matrix:")
print(header)
for i, row in enumerate(test_cm):
    print(f"Actual {i+1} " + " ".join(f"{v:>8d}" for v in row))

print(f"\nFirst {SHOW_FIRST_N_PROB_ROWS} test class-probability rows:")
print(y_test_proba[:SHOW_FIRST_N_PROB_ROWS])

print(f"\n{'='*60}")
print("TEST RARE-CLASS FOCUS")
print(f"{'='*60}")
for name in RARE_CLASS_NAMES:
    r = test_report_dict[name]
    print(f"{name}")
    print(f" recall : {r['recall']:.4f}")
    print(f" f1-score : {r['f1-score']:.4f}")
    print(f" precision : {r['precision']:.4f}")
print(f"{'='*60}")

# Keep names expected by later cells if needed
y_pred = y_test_pred
y_proba = y_test_proba


Evaluating BEST selected model on validation set...
Evaluating BEST selected model on held-out test set...

PERFORMANCE SUMMARY
Architecture : Weak (32,) ← fixed for this search
Best combo selected from validation search:
 MLP_MAX_ITER : 50
 N_ESTIMATORS : 250
 LEARNING_RATE : 0.5
 MLP alpha (L2) : 0.001
 MLP batch_size : 64
Training images : 15951
Validation images : 1989
Test images : 2060
Validation accuracy : 1.0000
Test accuracy : 0.8238
Validation Macro F1 : 1.0000
Test Macro F1 : 0.8233

VALIDATION REPORT
                           precision    recall  f1-score   support

Tissue Density Category 1       1.00      1.00      1.00       572
Tissue Density Category 2       1.00      1.00      1.00       496
Tissue Density Category 3       1.00      1.00      1.00       497
Tissue Density Category 4       1.00      1.00      1.00       424

                 accuracy                           1.00      1989
                macro avg       1.00      1.00      1.00      1989
          

In [6]:
# ============================================================
# CELL 6 — CLASS CONFIDENCE SNAPSHOT
# ============================================================
print("\nSample class-probability breakdown (first 10 test images):\n")

header = (
    f"{'Image':<8} | "
    f"{'Cat 1':>6} {'Cat 2':>6} {'Cat 3':>6} {'Cat 4':>6} | "
    f"{'Pred':>5} {'True':>5}"
)
print(header)
print("-" * len(header))

for i in range(min(10, len(X_test))):
    p_row = y_proba[i]
    pred  = y_pred[i]
    true  = y_test[i]
    match = "✅" if pred == true else "❌"

    print(
        f"{i+1:<8} | "
        f"{p_row[0]:>6.3f} {p_row[1]:>6.3f} {p_row[2]:>6.3f} {p_row[3]:>6.3f} | "
        f"{pred:>5} {true:>5}  {match}"
    )


Sample class-probability breakdown (first 10 test images):

Image    |  Cat 1  Cat 2  Cat 3  Cat 4 |  Pred  True
----------------------------------------------------
1        |  0.309  0.248  0.222  0.222 |     1     1  ✅
2        |  0.274  0.282  0.222  0.222 |     2     1  ❌
3        |  0.301  0.255  0.222  0.222 |     1     1  ✅
4        |  0.322  0.236  0.221  0.221 |     1     1  ✅
5        |  0.331  0.228  0.220  0.220 |     1     1  ✅
6        |  0.281  0.274  0.222  0.222 |     1     1  ✅
7        |  0.328  0.231  0.220  0.220 |     1     1  ✅
8        |  0.297  0.259  0.222  0.222 |     1     1  ✅
9        |  0.321  0.237  0.221  0.221 |     1     1  ✅
10       |  0.300  0.256  0.222  0.222 |     1     1  ✅


In [7]:

# ============================================================
# CELL 7 — SAVE MODEL
# ============================================================
print(f"\nSaving model pipeline → {MODEL_SAVE_PATH}")
joblib.dump(final_pipeline, MODEL_SAVE_PATH)
print(f" Done! AdaBoost + MLP tissue density model saved to {MODEL_SAVE_PATH}")
print(f" Benchmark-ready: checkpoints/best_model.pt")


Saving model pipeline → checkpoints/best_model_v2_2.pt
 Done! AdaBoost + MLP tissue density model saved to checkpoints/best_model_v2_2.pt
 Benchmark-ready: checkpoints/best_model.pt


In [8]:
# ==========================================
# CELL 8:AUTO-EVALUATE BENCHMARK
# ==========================================
print("Starting benchmark evaluation v2.2...")
!python3 /home/intern/Downloads/benchmark_v2_2_eval.py

print("\nRunning verifier...")
!python3 /home/intern/Downloads/verifier.py

Starting benchmark evaluation v2.2...
No arguments provided. Using default paths:
 Input : /home/intern/Downloads/benchmark_test_public.csv
 Output: /home/intern/Downloads/benchmark_test_v2_2_evaluated.csv


 BATCH CSV EVALUATION MODE (v2.2)
Using device : cuda
Loaded 800 image paths from CSV.
Loading ConvNeXt feature extractor...
Loading AdaBoost MLP Ensemble v2.2...

Starting batched inference...
Processing CSV: 100%|████████████████████████| 100/100 [01:25<00:00,  1.17it/s]

 Successfully predicted : 800 images
 Output saved to        : /home/intern/Downloads/benchmark_test_v2_2_evaluated.csv


Running verifier...
Total rows: 800
Error predictions: 0
Accuracy including ERROR: 67.25% (538/800)
Accuracy ignoring ERROR: 67.25% (538/800)
